# ⚙️ 02 — Feature Engineering
**Customer Churn Prediction · ChurnGuard**

We create 6 new domain-driven features and validate them visually.

---
**New features**
| Feature | Formula | Rationale |
|---------|---------|----------|
| `AvgCharges` | MonthlyCharges / (tenure + 1) | Spend per month |
| `TenureGroup` | Bucket 0–3 | Life-cycle stage |
| `ServiceCount` | # add-ons | Bundle loyalty |
| `ChargeRatio` | MonthlyCharges / (TotalCharges + 1) | New vs mature |
| `HasHighValuePlan` | Contract ∈ {1yr, 2yr} | Contract loyalty |
| `IsSeniorAlone` | Senior ∧ ¬Partner ∧ ¬Dependents | High-risk cohort |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_loader         import load_raw_data
from preprocess          import clean
from feature_engineering import engineer_features

plt.rcParams.update({
    'figure.facecolor': '#0F1117', 'axes.facecolor': '#1A1D27',
    'axes.edgecolor': '#2D3142', 'axes.labelcolor': '#E2E8F0',
    'xtick.color': '#94A3B8', 'ytick.color': '#94A3B8',
    'text.color': '#E2E8F0', 'grid.color': '#2D3142',
})
TEAL, CORAL, PURPLE, AMBER = '#00C2CB', '#FF6B6B', '#8B5CF6', '#F59E0B'
print('Libraries loaded.')

## 1. Load & clean raw data

In [ ]:
raw = load_raw_data()
df_clean = clean(raw)
print('Shape after cleaning:', df_clean.shape)
df_clean.head(3)

## 2. Apply feature engineering

In [ ]:
df = engineer_features(df_clean)
new_cols = ['AvgCharges', 'TenureGroup', 'ServiceCount',
            'ChargeRatio', 'HasHighValuePlan', 'IsSeniorAlone']

print('New features created:')
df[new_cols].describe().T.round(3)

## 3. Feature validation — AvgCharges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution
for label, color, data in [
    ('Retained', TEAL, df[df['Churn']==0]['AvgCharges']),
    ('Churned',  CORAL, df[df['Churn']==1]['AvgCharges']),
]:
    axes[0].hist(data, bins=30, alpha=0.7, color=color, label=label, edgecolor='none')
axes[0].set_title('AvgCharges Distribution by Churn', fontweight='bold')
axes[0].set_xlabel('AvgCharges')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Box
groups = [df[df['Churn']==0]['AvgCharges'], df[df['Churn']==1]['AvgCharges']]
bp = axes[1].boxplot(groups, patch_artist=True,
                     medianprops=dict(color='white', linewidth=2))
for patch, clr in zip(bp['boxes'], [TEAL, CORAL]):
    patch.set_facecolor(clr); patch.set_alpha(0.75)
axes[1].set_xticklabels(['Retained', 'Churned'])
axes[1].set_title('AvgCharges vs Churn (Boxplot)', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visuals/06_avg_charges.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print('AvgCharges — Retained:', df[df['Churn']==0]['AvgCharges'].mean().round(2))
print('AvgCharges — Churned: ', df[df['Churn']==1]['AvgCharges'].mean().round(2))

## 4. Tenure group vs churn

In [ ]:
tg_labels = {0:'New\n(0-12mo)', 1:'Growing\n(1-2yr)', 2:'Loyal\n(2-4yr)', 3:'Champion\n(4+yr)'}
df['TGLabel'] = df['TenureGroup'].map(tg_labels)

churn_rates = df.groupby('TGLabel')['Churn'].mean() * 100
order = list(tg_labels.values())
churn_rates = churn_rates.reindex(order, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4))
colors = [CORAL if v > 30 else AMBER if v > 20 else TEAL for v in churn_rates.values]
bars = ax.bar(churn_rates.index, churn_rates.values, color=colors, width=0.55, edgecolor='none')
for bar, val in zip(bars, churn_rates.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Churn Rate by Tenure Group', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../visuals/07_tenure_group.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 5. ServiceCount vs churn

In [ ]:
svc_churn = df.groupby('ServiceCount')['Churn'].mean() * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Churn rate by service count
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(svc_churn)))
bars = ax1.bar(svc_churn.index, svc_churn.values, color=colors, width=0.7)
for bar, val in zip(bars, svc_churn.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax1.set_title('Churn Rate by Service Count', fontweight='bold')
ax1.set_xlabel('Number of Services Subscribed')
ax1.set_ylabel('Churn Rate (%)')
ax1.grid(axis='y', alpha=0.3)

# Count of customers
svc_count = df['ServiceCount'].value_counts().sort_index()
ax2.bar(svc_count.index, svc_count.values, color=TEAL, alpha=0.8, width=0.7)
ax2.set_title('Customer Count by Service Count', fontweight='bold')
ax2.set_xlabel('Number of Services Subscribed')
ax2.set_ylabel('Customers')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visuals/08_service_count.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 6. Feature importance preview (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose  import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from preprocess import CATEGORICAL_COLS
from feature_engineering import ENGINEERED_NUMERIC_COLS, ENGINEERED_PASSTHROUGH_COLS

X = df.drop(columns=['Churn', 'TGLabel'], errors='ignore')
y = df['Churn'].astype(int)

NUM   = [c for c in ENGINEERED_NUMERIC_COLS if c in X.columns]
CAT   = [c for c in CATEGORICAL_COLS if c in X.columns]
PASS  = [c for c in ENGINEERED_PASSTHROUGH_COLS if c in X.columns]

prep = ColumnTransformer([
    ('num', StandardScaler(), NUM),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), CAT),
], remainder='passthrough')

rf = Pipeline([('prep', prep), ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))])
rf.fit(X, y)

importances = rf.named_steps['clf'].feature_importances_
enc = rf.named_steps['prep'].named_transformers_['cat']
cat_names = list(enc.get_feature_names_out(CAT))
all_features = NUM + cat_names + PASS

imp_df = (
    pd.DataFrame({'feature': all_features[:len(importances)], 'importance': importances})
    .nlargest(15, 'importance').sort_values('importance')
)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = [TEAL if f in new_cols else '#4B5563' for f in imp_df['feature']]
bars = ax.barh(imp_df['feature'], imp_df['importance'], color=colors_bar)
for bar, val in zip(bars, imp_df['importance']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_title('Top 15 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')

from matplotlib.patches import Patch
legend_elems = [Patch(facecolor=TEAL, label='Engineered Feature'),
                Patch(facecolor='#4B5563', label='Original Feature')]
ax.legend(handles=legend_elems)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../visuals/09_feature_importance_preview.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

print('Engineered features in top 15:')
for f in imp_df['feature']:
    if f in new_cols:
        print(f'  ✅ {f}')

## Summary

All 6 engineered features show predictive signal:
- **AvgCharges** separates churn groups effectively
- **ServiceCount** shows clear negative correlation with churn
- **TenureGroup** captures non-linear tenure effects
- **HasHighValuePlan** is among top predictors

➡️ Proceed to **03_model_training.ipynb**